# Introdução à regressão logística

[▶ Abrir este notebook no Google Colab](https://colab.research.google.com/github/lalvim/disciplina_computacao_aplicada_humanidades_digitais/blob/main/unidade_07/04_regressao_logistica.ipynb)

Quando a resposta registra ocorrência — sim/não, presente/ausente — modelamos a
probabilidade do evento. O exemplo pergunta: **quais características estão
associadas à presença de biblioteca pública no conjunto fictício?**

![Probabilidade transforma-se em chance e logaritmo da chance antes da combinação linear.](imagens/04_probabilidade_chance.svg)

In [ ]:
# @title Preparação do ambiente — execute esta célula no Google Colab
from pathlib import Path
import importlib.util
import os
import subprocess
import sys

URL_REPOSITORIO = 'https://github.com/lalvim/disciplina_computacao_aplicada_humanidades_digitais.git'
REPOSITORIO = Path(
    "/content/disciplina_computacao_aplicada_humanidades_digitais"
)
PASTA_UNIDADE = REPOSITORIO / 'unidade_07'

try:
    import google.colab  # type: ignore  # noqa: F401
    EM_COLAB = True
except ImportError:
    EM_COLAB = False

if EM_COLAB:
    if not (REPOSITORIO / ".git").exists():
        subprocess.run(
            [
                "git",
                "clone",
                "--depth",
                "1",
                "--branch",
                "main",
                URL_REPOSITORIO,
                str(REPOSITORIO),
            ],
            check=True,
        )

    PACOTES_COLAB = []
    ausentes = [
        especificacao
        for modulo, especificacao in PACOTES_COLAB
        if importlib.util.find_spec(modulo) is None
    ]
    if ausentes:
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q", *ausentes],
            check=True,
        )

    os.chdir(PASTA_UNIDADE)
    print("Ambiente preparado em:", Path.cwd())
else:
    print("Ambiente local: nenhuma clonagem necessária.")

## 1. Probabilidade, chance e logito

A probabilidade $p$ varia de 0 a 1. A chance compara ocorrência e não ocorrência:

$$chance = \frac{p}{1-p}$$

Se $p=0{,}75$, a chance é 3:1. O logito transforma a chance numa escala sem limites:

$$\log\left(\frac{p_i}{1-p_i}\right)=\beta_0+\beta_1X_{1i}+\cdots+\beta_kX_{ki}$$

Um coeficiente exponenciado, $e^{\beta}$, é uma razão de chances. Razão de chances
não é a mesma coisa que diferença de probabilidades.

A próxima célula mostra didaticamente a transformação dos dados e a otimização. Não
se espera que você reprograme o algoritmo; concentre-se em como a resposta é
codificada, como as escalas são padronizadas e como a saída será traduzida.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

dados = pd.read_csv("dados/municipios_culturais.csv")
y_logistico = (dados["biblioteca_publica"] == "sim").astype(float).to_numpy()
variaveis_logisticas = ["renda_pc", "urbanizacao_pct", "escolaridade_media_anos", "distancia_capital_km"]
medias = dados[variaveis_logisticas].mean()
desvios = dados[variaveis_logisticas].std(ddof=0)
X_padronizado = ((dados[variaveis_logisticas] - medias) / desvios).to_numpy()
X_logistico = np.column_stack([np.ones(len(dados)), X_padronizado])

def sigmoide(z):
    z = np.clip(z, -30, 30)
    return 1 / (1 + np.exp(-z))

def ajustar_logistico(X, y, taxa=.08, iteracoes=8000):
    beta = np.zeros(X.shape[1])
    for _ in range(iteracoes):
        probabilidades = sigmoide(X @ beta)
        beta -= taxa * (X.T @ (probabilidades - y)) / len(y)
    return beta, sigmoide(X @ beta)

beta_logistico, probabilidades = ajustar_logistico(X_logistico, y_logistico)
coeficientes_logisticos = pd.DataFrame({
    "termo": ["intercepto", *[f"{v} (+1 desvio-padrão)" for v in variaveis_logisticas]],
    "coeficiente_logito": beta_logistico,
    "razao_de_chances": np.exp(beta_logistico),
})
display(coeficientes_logisticos.round(3))

## 2. Por que padronizar neste exemplo?

Renda, quilômetros e anos têm escalas diferentes. A padronização faz cada
coeficiente representar aumento de um desvio-padrão e estabiliza a otimização
numérica didática. O sinal mostra a direção da associação condicionada; a razão de
chances mostra a multiplicação estimada das chances, mantidos os demais termos.

A implementação acima explicita o algoritmo para aprendizagem. Em pesquisa,
bibliotecas estatísticas devem fornecer erros-padrão, intervalos e testes; eles não
são substituídos por esta demonstração.

## 3. Traduzir o modelo de volta para probabilidades

Razões de chances são pouco intuitivas. Vamos variar a urbanização, manter as
outras variáveis em suas médias e observar probabilidades previstas. Isso torna
claro que uma mesma razão de chances não implica a mesma variação percentual em
todos os pontos da curva.

In [ ]:
grade_urbana = np.linspace(dados["urbanizacao_pct"].min(), dados["urbanizacao_pct"].max(), 100)
cenarios_originais = pd.DataFrame({v: np.repeat(medias[v], len(grade_urbana)) for v in variaveis_logisticas})
cenarios_originais["urbanizacao_pct"] = grade_urbana
cenarios_z = ((cenarios_originais[variaveis_logisticas] - medias) / desvios).to_numpy()
prob_grade = sigmoide(np.column_stack([np.ones(len(grade_urbana)), cenarios_z]) @ beta_logistico)
plt.plot(grade_urbana, prob_grade, color="#a44a3f", linewidth=2)
plt.xlabel("Urbanização (%)"); plt.ylabel("Probabilidade prevista de biblioteca")
plt.ylim(0, 1); plt.title("Cenários do modelo — demais variáveis nas médias")
plt.tight_layout(); plt.show()

faixas = pd.qcut(probabilidades, 5, duplicates="drop")
calibracao_descritiva = pd.DataFrame({"faixa": faixas, "observado": y_logistico, "previsto": probabilidades}).groupby("faixa", observed=True).agg(n=("observado", "size"), proporcao_observada=("observado", "mean"), probabilidade_media=("previsto", "mean"))
display(calibracao_descritiva.round(3))

A tabela confronta probabilidades médias com proporções observadas por faixa. É um
diagnóstico descritivo interno, não prova de desempenho fora da amostra. Avaliação
preditiva com separação treino/teste pertence à Unidade 8.

### U07-A05 — Atividade — interpretar ocorrência

1. Converta as probabilidades 0,20 e 0,75 em chances.
2. Interprete uma razão de chances do modelo, mencionando a padronização.
3. Compare as probabilidades previstas em 50% e 90% de urbanização.
4. Escreva por que o resultado não demonstra que urbanização cria bibliotecas.

**Dinâmica (30 minutos):** individual até o item 2; os itens 3 e 4 são revisados em
duplas. Cada dupla substitui uma frase causal por uma formulação associativa.

**Resposta:** escreva aqui.

## Passagem

Já sabemos estimar, diagnosticar e traduzir dois tipos de modelo. A oficina não
pede um relatório novo: ela orienta a incorporação de uma análise pertinente ao
mesmo projeto cumulativo da disciplina.

**Leitura:** Hosmer, Lemeshow e Sturdivant (2013), Gelman, Hill e Vehtari
(2020) e Hernán e Robins (2020), listados em `referencias.md`.